# Practice: J2 node regression

A sandbox for `orbit_correction.truth_prop`. Earth's oblateness (J2) makes an orbit's plane
precess: the ascending node drifts westward for prograde orbits, eastward for retrograde
ones, and not at all for polar ones. First-order theory:

Ω̇ = −(3/2) n J2 (Rₑ / p)² cos i

This notebook propagates circular orbits with the **project's truth propagator, gravity only**
(zonal degree 2, drag off), measures Ω̇ from the trajectory and compares it with the formula.
Then it changes altitude and inclination, and finally turns the higher zonal terms back on.

Needs the datagen group (`uv sync --group datagen`); the first cell starts a JVM in the
kernel (a few seconds). Nothing here writes to `src/` or `data/`.

In [ ]:
import dataclasses

import numpy as np

from orbit_correction import truth_prop as T
from orbit_correction.types import Trajectory

DAY = 86400.0
# Truth starts from one state here, so the epoch-state rule is named explicitly: the
# configured fitted start (2026-11-25) needs SGP4 samples before the epoch.
cfg = dataclasses.replace(T.load_truth_config(), start_rule=T.EPOCH_START_RULE)
ctx = T._context(cfg)  # starts this kernel's JVM (once; restart the kernel to change JVM options)
MU = ctx.mu            # m^3/s^2, EIGEN-6S
print("zonal degree in config:", cfg.zonal_degree, "| mu:", MU)

## Helpers

`circular_state` builds a circular-velocity GCRF state. `node_rate` propagates it for
`days` days, computes the right ascension of the node from the angular momentum at every
output time, and fits a straight line. Epoch 2000-01-02, so the GCRF pole and Earth's pole
agree to arcseconds and GCRF angles are the angles J2 acts on.

In [ ]:
T0 = DAY  # 2000-01-02T12:00 TT

def circular_state(alt_km, inc_deg, raan_deg=0.0):
    r = 6378.137 + alt_km
    v = np.sqrt(MU / 1e9 / r)
    i, o = np.radians(inc_deg), np.radians(raan_deg)
    pos = r * np.array([np.cos(o), np.sin(o), 0.0])
    vel = v * np.array([-np.sin(o) * np.cos(i), np.cos(o) * np.cos(i), np.sin(i)])
    return Trajectory(1, np.array([T0]), pos[None, :], vel[None, :], "practice")

def node_rate(alt_km, inc_deg, degree=2, days=3.0):
    """Measured and closed-form node rate in degrees per day."""
    c = dataclasses.replace(cfg, zonal_degree=degree, atmosphere="none")
    init = circular_state(alt_km, inc_deg)
    t = T.output_grid(T0, days * DAY, 60.0)
    tr = T.propagate_truth(init, None, t, c, None).trajectory
    h = np.cross(tr.r_gcrf_km, tr.v_gcrf_km_s)
    raan = np.unwrap(np.arctan2(h[:, 0], -h[:, 1]))
    measured = np.degrees(np.polyfit(t - T0, raan, 1)[0]) * DAY
    prov = ctx.gravity_provider(2)
    j2 = -np.sqrt(5.0) * prov.onDate(ctx.date(T0)).getNormalizedCnm(2, 0)
    a = (6378.137 + alt_km) * 1e3
    n = np.sqrt(MU / a**3)
    closed = np.degrees(-1.5 * n * j2 * (prov.getAe() / a) ** 2 * np.cos(np.radians(inc_deg))) * DAY
    return measured, closed

## Change the inclination (500 km)

Expect: largest westward drift at low inclination, zero at 90°, eastward beyond 90°.
The small measured-vs-formula gap comes from using the osculating radius as the mean
semi-major axis and from J2's short-period terms over only three days.

In [ ]:
print(f"{'inc (deg)':>9} {'measured':>10} {'formula':>10}  (deg/day)")
for inc in (28.5, 51.6, 70.0, 90.0, 97.4, 120.0):
    m, c = node_rate(500.0, inc)
    print(f"{inc:9.1f} {m:10.4f} {c:10.4f}")

## Change the altitude (51.6°)

The rate scales roughly as a^(−7/2): the effect weakens quickly with altitude.

In [ ]:
for alt in (300.0, 500.0, 800.0, 1200.0):
    m, c = node_rate(alt, 51.6)
    print(f"{alt:7.0f} km  measured {m:8.4f}  formula {c:8.4f} deg/day")

## Sun-synchronous orbits

A Sun-synchronous orbit's node turns eastward at 360°/365.2422 days = 0.9856°/day, so it
keeps the same angle to the Sun. Solve the formula for the inclination, then check it with
the propagator. Try other altitudes.

In [ ]:
alt = 500.0
target = 360.0 / 365.2422
_, per_cos = node_rate(alt, 0.0)          # formula at cos i = 1
inc_sso = np.degrees(np.arccos(target / per_cos))
m, c = node_rate(alt, inc_sso)
print(f"SSO inclination at {alt:.0f} km: {inc_sso:.3f} deg; measured node rate {m:.4f} deg/day")

## Turn on the higher zonals

The configured truth uses zonal degree `cfg.zonal_degree`. J3, J4, ... barely change the
node rate (J2 dominates by about 1000x); their effect on along-track position is larger,
which is what the zonal convergence check (`scripts/zonal_convergence.py`) measured.

In [ ]:
for degree in (2, 4, 8, cfg.zonal_degree):
    m, _ = node_rate(500.0, 51.6, degree=degree)
    print(f"degree {degree:3d}: {m:.5f} deg/day")

## Things to try

- Make the orbit eccentric (edit `circular_state`) and measure the perigee rotation ω̇;
  it vanishes at the critical inclination 63.4°.
- Propagate longer (`days=10`) and watch the measured rate settle onto the formula.
- Copy `node_rate` and add drag back (`atmosphere="NRLMSISE-00"` needs a space-weather copy:
  see `practice/drag_horizon.ipynb`).